# Lab 18: Computational Complexity and Eigenvalue Computation

This independent-study notebook contains explanations, code, solutions, and similar practice questions. The main goals are to compare operation counts, use sparse matrices, implement the power method, compute Rayleigh quotients, and interpret convergence.


## 1. Cost comparison: $(AB)x$ versus $A(Bx)$

For $A\in\mathbb R^{m\times n}$, $B\in\mathbb R^{n\times p}$, and $x\in\mathbb R^p$,

$$
(AB)x=A(Bx).
$$

But the costs are different:

$$
(AB)x: O(mnp)+O(mp),\qquad A(Bx):O(np)+O(mn).
$$


In [ ]:
def cost_form_then_multiply(m, n, p):
    return m*n*p + m*p

def cost_vector_first(m, n, p):
    return n*p + m*n

m = n = p = 1000
c1 = cost_form_then_multiply(m,n,p)
c2 = cost_vector_first(m,n,p)
print("form AB first:", c1)
print("multiply vector first:", c2)
print("ratio:", c1/c2)


### Answer

For $m=n=p=1000$, forming $AB$ first takes about $10^9$ operations, while computing two matrix-vector products takes about $2\cdot 10^6$ operations. The vector-first method is much cheaper for one vector.

### Similar practice

Repeat the comparison for $m=2000$, $n=500$, and $p=1000$.


In [ ]:
m, n, p = 2000, 500, 1000
c1 = cost_form_then_multiply(m,n,p)
c2 = cost_vector_first(m,n,p)
print(c1, c2, c1/c2)


## 2. Sparse matrix storage

A dense $n\times n$ matrix stores $n^2$ entries. A tridiagonal matrix has approximately $3n-2$ nonzero entries.


In [ ]:
import numpy as np
from scipy import sparse

n = 10_000
main = 2*np.ones(n)
off = -1*np.ones(n-1)
A = sparse.diags([off, main, off], [-1,0,1], format="csr")

print("shape:", A.shape)
print("nonzeros:", A.nnz)
print("dense entries:", n*n)
print("density:", A.nnz/(n*n))


### Answer

The tridiagonal matrix has only $3n-2$ nonzero entries. For $n=10{,}000$, the density is about $0.0003$, or $0.03\%$.

### Similar practice

Estimate the density for $n=1{,}000{,}000$.


In [ ]:
n = 1_000_000
nnz = 3*n - 2
density = nnz/(n*n)
print("nnz:", nnz)
print("density:", density)


## 3. Implement the normalized power method

The normalized power method is

$$
y_{k+1}=Ax_k,\qquad x_{k+1}=\frac{y_{k+1}}{\|y_{k+1}\|}.
$$


In [ ]:
def power_method(A, x0=None, steps=20):
    A = np.array(A, dtype=float)
    n = A.shape[0]
    if x0 is None:
        x = np.ones(n)
    else:
        x = np.array(x0, dtype=float)
    x = x/np.linalg.norm(x)
    history = []
    for k in range(steps):
        y = A @ x
        if np.linalg.norm(y) == 0:
            raise ValueError("Iteration hit the zero vector.")
        x = y/np.linalg.norm(y)
        lam = (x @ A @ x)/(x @ x)
        history.append(lam)
    return x, np.array(history)

A = np.array([[4,0],[0,1]], dtype=float)
x, hist = power_method(A, x0=[1,1], steps=10)
print("final direction:", x)
print("Rayleigh history:", hist)
print("true eigenvalues:", np.linalg.eigvals(A))


### Answer

The dominant eigenvalue is $4$, with eigendirection $(1,0)^T$. Starting from $(1,1)^T$, the normalized iterates converge to the $x$-axis.

### Similar practice

Use $A=\operatorname{diag}(5,2)$ and $x_0=(1,3)^T$. Predict the limiting direction before running the code.


In [ ]:
A = np.diag([5,2])
x, hist = power_method(A, x0=[1,3], steps=12)
print("final direction:", x)
print("last Rayleigh quotient:", hist[-1])


## 4. Slow convergence

The convergence rate is controlled by

$$
\left|\frac{\lambda_2}{\lambda_1}\right|.
$$

If this ratio is close to $1$, convergence is slow.


In [ ]:
A_fast = np.diag([10, 1])
A_slow = np.diag([10, 9])

_, h_fast = power_method(A_fast, x0=[1,1], steps=20)
_, h_slow = power_method(A_slow, x0=[1,1], steps=20)

print("fast final Rayleigh:", h_fast[-1])
print("slow final Rayleigh:", h_slow[-1])
print("slow history last five:", h_slow[-5:])


### Answer

For $\operatorname{diag}(10,1)$, the ratio is $0.1$, so convergence is fast. For $\operatorname{diag}(10,9)$, the ratio is $0.9$, so convergence is slow.

### Similar practice

Try $\operatorname{diag}(10,9.9)$ and observe the slower convergence.


In [ ]:
A_very_slow = np.diag([10, 9.9])
_, h = power_method(A_very_slow, x0=[1,1], steps=50)
print("last Rayleigh quotient:", h[-1])
print("true largest eigenvalue:", 10)


## 5. Rayleigh quotient

For symmetric $A$, the Rayleigh quotient is

$$
R_A(x)=\frac{x^TAx}{x^Tx}.
$$


In [ ]:
def rayleigh(A, x):
    A = np.array(A, dtype=float)
    x = np.array(x, dtype=float)
    return (x @ A @ x)/(x @ x)

A = np.array([[2,1],[1,2]], dtype=float)
print(rayleigh(A, [1,0]))
print(rayleigh(A, [1,1]))
print(np.linalg.eigvalsh(A))


### Answer

For $x=(1,0)^T$, the Rayleigh quotient is $2$. For $x=(1,1)^T$, it is $3$, which is the largest eigenvalue.

### Similar practice

Compute the Rayleigh quotient for $x=(1,-1)^T$.


In [ ]:
print(rayleigh(A, [1,-1]))


## 6. Deflation

If $A$ is symmetric, $v_1$ is a unit eigenvector, and $Av_1=\lambda_1v_1$, then

$$
B=A-\lambda_1v_1v_1^T
$$

removes that eigendirection.


In [ ]:
A = np.array([[3,1],[1,2]], dtype=float)
vals, vecs = np.linalg.eigh(A)
idx = np.argsort(vals)[::-1]
vals = vals[idx]
vecs = vecs[:, idx]

lam1 = vals[0]
v1 = vecs[:,0]
B = A - lam1*np.outer(v1,v1)

print("eigenvalues of A:", vals)
print("B =\n", B)
print("eigenvalues of B:", np.linalg.eigvalsh(B))


### Answer

The deflated matrix has one eigenvalue near $0$ and keeps the remaining eigenvalue. Small numerical roundoff errors may appear.


## 7. Reflection questions

1. Why is it often better to compute $A(Bx)$ instead of $(AB)x$?
2. Why is sparse storage important?
3. What controls the convergence rate of the power method?
4. Why does the Rayleigh quotient give an eigenvalue when the input vector is an eigenvector?
5. What is one limitation of deflation?

### Suggested answers

1. It avoids forming a large intermediate matrix.
2. Sparse storage uses memory proportional to the number of nonzero entries rather than all entries.
3. The ratio $|\lambda_2/\lambda_1|$.
4. If $Av=\lambda v$, then $v^TAv/(v^Tv)=\lambda$.
5. Errors in one computed eigenvector can affect later computations.
